# Adaptive RAG - Kaggle 启动脚本
运行后将通过 ngrok 暴露 Web 界面，可测试文件上传和对话功能

In [ ]:
# 1. 安装依赖
!pip install -q pyngrok

In [ ]:
# 2. 设置 ngrok Token（从 Kaggle Secrets 读取）
# 在 Kaggle Notebook 右上角 Add-ons → Secrets 添加 NGROK_AUTH_TOKEN
# 免费注册: https://dashboard.ngrok.com/signup
import os
from google.colab import userdata

try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    os.environ['NGROK_AUTH_TOKEN'] = NGROK_AUTH_TOKEN
    print('✅ ngrok token 已加载')
except Exception:
    # 也可以手动输入
    NGROK_AUTH_TOKEN = input('请输入 ngrok authtoken（https://dashboard.ngrok.com 获取）: ').strip()
    os.environ['NGROK_AUTH_TOKEN'] = NGROK_AUTH_TOKEN

In [ ]:
# 3. 后台启动 FastAPI 服务器
import threading
import uvicorn
import sys

sys.path.append('/kaggle/working')

def run_server():
    # 设置环境变量抑制警告
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    os.environ['ABSL_MIN_LOG_LEVEL'] = '3'
    
    # 导入并初始化 RAG 系统（会启动 server.py）
    from server import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

threading.Thread(target=run_server, daemon=True).start()
print('⏳ 服务器启动中，等待 10 秒...')
import time
time.sleep(10)
print('✅ 服务器已启动')

In [ ]:
# 4. 创建 ngrok 隧道
from pyngrok import ngrok, conf

# 设置 ngrok 配置
conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']

# 关闭旧隧道
ngrok.kill()

# 创建新隧道
public_url = ngrok.connect(8000, bind_tls=True).public_url
print(f'\n' + '='*60)
print(f'🌐 公开访问地址: {public_url}')
print(f'='*60)
print(f'\n📤 上传文件: {public_url}')
print(f'💬 开始对话: {public_url}')
print(f'📋 API 文档: {public_url}/docs')
print(f'\n⚠️ 注意: 此链接仅在 Kaggle Notebook 运行时有效')

In [ ]:
# 5. (可选) 测试 API 是否正常
import requests
try:
    r = requests.get(f'{public_url}/api/health', timeout=5)
    print(f'✅ 健康检查: {r.json()}')
except Exception as e:
    print(f'⚠️ 健康检查失败: {e}')